In [ ]:
"""
=============================================================================
 ÉTUDE 2 — COMPARAISON DES SOURCES DU SIGNAL CALCIQUE
=============================================================================
 Question : quelle source produit le signal calcique le plus discriminant,
            le U-Net de segmentation (M2, Dice = 0.784) ou le classifieur
            CNN (M3, F1 = 0.369) ?

 Méthode  : la composante calcique de la formule déployée V5 est remplacée
            successivement par chaque source candidate, les trois autres
            pondérations restant inchangées. Seule la source varie.

 Entrée   : resultats_phase4_v5.csv
 GPU      : non nécessaire
=============================================================================
"""

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
CSV_PATH = '/kaggle/input/datasets/abdallahouahmane/resultas-phase4/resultats_phase4_v5.csv'

W = {'ctr': 0.15, 'cong': 0.40, 'calcif': 0.10, 'cardio': 0.35}
SEUIL_M2 = 0.5          # % de surface au-delà duquel M2 est jugé positif
DICE_M2  = 0.784
F1_M3    = 0.369

NOM_M3   = 'M3 CNN\n(source déployée)'
NOM_SANS = 'Aucun signal calcique'

print("=" * 62)
print("  ÉTUDE 2 — SOURCES DU SIGNAL CALCIQUE")
print("=" * 62)

df = pd.read_csv(CSV_PATH)
df['y_true'] = df['label_reel'].apply(
    lambda x: 1 if ('Cardiomegaly' in str(x) or 'Edema' in str(x)) else 0)

print(f"\n  CSV chargé : {len(df)} images")
print(f"  Positifs : {df['y_true'].sum()} | "
      f"Négatifs : {(df['y_true'] == 0).sum()}")


def score_v5(source_calcique):
    """Formule V5, la composante calcique étant paramétrable."""
    return (W['ctr'] * df['s_ctr'] + W['cong'] * df['s_cong']
            + W['calcif'] * source_calcique + W['cardio'] * df['prob_cardio'])


# ============================================================
# 1. STATISTIQUES DES DEUX SOURCES
# ============================================================
print(f"\n{'='*62}")
print("  CARACTÉRISATION DES SOURCES")
print(f"{'='*62}\n")

print(f"  M2 — surface segmentée (calcif_pct, en %)")
print(f"    min {df['calcif_pct'].min():.3f} | max {df['calcif_pct'].max():.3f} "
      f"| moyenne {df['calcif_pct'].mean():.3f}")
print(f"    non nul sur {(df['calcif_pct'] > 0).sum()} images "
      f"({(df['calcif_pct'] > 0).mean()*100:.1f} %)")

print(f"\n  M3 — probabilité de classe calcique (s_calcif)")
print(f"    min {df['s_calcif'].min():.3f} | max {df['s_calcif'].max():.3f} "
      f"| moyenne {df['s_calcif'].mean():.3f}")
print(f"    {df['s_calcif'].nunique()} valeurs distinctes")

# --- Pouvoir discriminant brut de chaque source ---
auc_brut_m2 = roc_auc_score(df['y_true'], df['calcif_pct'])
auc_brut_m3 = roc_auc_score(df['y_true'], df['s_calcif'])
print(f"\n  AUC de la source brute, hors fusion :")
print(f"    M2 (calcif_pct) : {auc_brut_m2:.4f}")
print(f"    M3 (s_calcif)   : {auc_brut_m3:.4f}")

# ============================================================
# 2. NORMALISATIONS DE M2
# ============================================================
df['m2_lin']  = (df['calcif_pct'] / 2.0).clip(0, 1)
df['m2_sqrt'] = np.sqrt((df['calcif_pct'] / 5.0).clip(0, 1))
df['m2_bin']  = (df['calcif_pct'] > SEUIL_M2).astype(float)

print(f"\n  Normalisations de M2 :")
print(f"    linéaire (pct/2)   : moyenne {df['m2_lin'].mean():.3f}")
print(f"    racine (√(pct/5))  : moyenne {df['m2_sqrt'].mean():.3f}")
print(f"    binaire (> {SEUIL_M2} %)  : {int(df['m2_bin'].sum())} images positives")

# ============================================================
# 3. CONFIGURATIONS
# ============================================================
configurations = {
    NOM_M3: (
        df['s_calcif'],
        'Composante calcique = probabilité CNN M3'),
    'M2 linéaire\n(pct / 2)': (
        df['m2_lin'],
        'Composante calcique = calcif_pct / 2'),
    'M2 racine\n(√(pct / 5))': (
        df['m2_sqrt'],
        'Composante calcique = racine de calcif_pct / 5'),
    f'M2 binaire\n(> {SEUIL_M2} %)': (
        df['m2_bin'],
        f'Composante calcique = 1 si calcif_pct > {SEUIL_M2} %'),
    'M2 + M3\n(60 / 40)': (
        0.60 * df['m2_lin'] + 0.40 * df['s_calcif'],
        'Combinaison pondérée des deux sources'),
    'M2 + M3\n(50 / 50)': (
        0.50 * df['m2_lin'] + 0.50 * df['s_calcif'],
        'Moyenne des deux sources'),
    NOM_SANS: (
        pd.Series(0.0, index=df.index),
        'Composante calcique nulle'),
}

# ============================================================
# 4. AUC PAR CONFIGURATION
# ============================================================
print(f"\n{'='*62}")
print("  AUC — LA SOURCE CALCIQUE VARIE, LE RESTE DE V5 EST FIXE")
print(f"{'='*62}\n")
print(f"  {'Source calcique':<28} {'AUC':>8} {'Δ vs M3':>10}")
print("  " + "-" * 48)

resultats = {}
for nom, (source, description) in configurations.items():
    scores = score_v5(source)
    resultats[nom] = {'auc': round(roc_auc_score(df['y_true'], scores), 4),
                      'scores': scores, 'description': description}

auc_m3   = resultats[NOM_M3]['auc']
auc_sans = resultats[NOM_SANS]['auc']

for nom, v in resultats.items():
    delta = v['auc'] - auc_m3
    marque = '  ← déployée' if nom == NOM_M3 else ''
    print(f"  {nom.replace(chr(10), ' '):<28} {v['auc']:>8.4f} "
          f"{delta:>+10.4f}{marque}")

# ============================================================
# 5. ANALYSE
# ============================================================
print(f"\n{'='*62}")
print("  ANALYSE")
print(f"{'='*62}\n")

meilleur_m2 = max(
    ((n, v) for n, v in resultats.items() if n.startswith('M2 ') and '+' not in n),
    key=lambda x: x[1]['auc'])

print(f"  Source déployée (M3)        : {auc_m3:.4f}")
print(f"  Meilleure variante de M2    : {meilleur_m2[1]['auc']:.4f}  "
      f"({meilleur_m2[0].replace(chr(10), ' ')})")
print(f"  Sans composante calcique    : {auc_sans:.4f}")
print(f"\n  Apport de M3 sur l'absence de signal : {auc_m3 - auc_sans:+.4f}")
print(f"  Écart entre M3 et la meilleure variante de M2 : "
      f"{auc_m3 - meilleur_m2[1]['auc']:+.4f}")

if auc_m3 > meilleur_m2[1]['auc']:
    print(f"\n  M3 domine toutes les normalisations de M2 testées.")
if auc_m3 > auc_sans:
    print(f"  La composante calcique contribue positivement au score V5.")

# ============================================================
# 6. POURQUOI M2 ÉCHOUE — SIGNAL PAR GROUPE
# ============================================================
print(f"\n{'='*62}")
print("  SIGNAL DES DEUX SOURCES PAR GROUPE CLINIQUE")
print(f"{'='*62}\n")

groupes = {
    'Normal':        df[df['label_reel'] == 'No Finding'],
    'Cardiomégalie': df[df['label_reel'].str.contains('Cardiomegaly', na=False)],
    'Œdème':         df[df['label_reel'].str.contains('Edema', na=False)],
}

print(f"  {'Groupe':<16} {'M2 moy (%)':>12} {'M2 > seuil':>12} "
      f"{'M3 moy':>10}")
print("  " + "-" * 52)
for nom, sub in groupes.items():
    if len(sub):
        print(f"  {nom:<16} {sub['calcif_pct'].mean():>11.3f}% "
              f"{(sub['calcif_pct'] > SEUIL_M2).mean()*100:>11.1f}% "
              f"{sub['s_calcif'].mean():>10.3f}")

etendue_m2 = (max(g['calcif_pct'].mean() for g in groupes.values())
              - min(g['calcif_pct'].mean() for g in groupes.values()))
etendue_m3 = (max(g['s_calcif'].mean() for g in groupes.values())
              - min(g['s_calcif'].mean() for g in groupes.values()))
print(f"\n  Étendue inter-groupes : M2 = {etendue_m2:.3f} point de %, "
      f"M3 = {etendue_m3:.3f}")
print(f"  La surface segmentée par M2 varie peu d'un groupe à l'autre,")
print(f"  ce qui explique son faible pouvoir discriminant.")

# ============================================================
# 7. VISUALISATION
# ============================================================
fig = plt.figure(figsize=(21, 13))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.32)

noms = [n.replace('\n', ' ') for n in resultats.keys()]
aucs = [v['auc'] for v in resultats.values()]

couleurs = []
for nom in resultats.keys():
    if nom == NOM_M3:
        couleurs.append('#C0392B')
    elif nom == NOM_SANS:
        couleurs.append('#7F8C8D')
    elif '+' in nom:
        couleurs.append('#F39C12')
    else:
        couleurs.append('#1D9E75')

marge = 0.02
ax1 = fig.add_subplot(gs[0, :2])
bars = ax1.barh(noms, aucs, color=couleurs, alpha=0.88, edgecolor='white')
ax1.axvline(auc_m3, color='#C0392B', ls='--', lw=2,
            label=f'M3, source déployée ({auc_m3:.4f})')
ax1.axvline(auc_sans, color='#7F8C8D', ls=':', lw=1.8,
            label=f'Sans signal calcique ({auc_sans:.4f})')
ax1.set_xlim(min(aucs) - marge, max(aucs) + marge)
ax1.set_xlabel('AUC-ROC', fontsize=11)
ax1.set_title('Étude 2 — AUC de la formule V5 selon la source calcique\n'
              'Rouge : M3 | Vert : variantes de M2 | '
              'Orange : combinaisons | Gris : aucun signal',
              fontweight='bold', fontsize=11)
ax1.legend(fontsize=9, loc='lower right')
ax1.grid(alpha=0.3, axis='x')
for bar, a in zip(bars, aucs):
    ax1.text(a + marge * 0.06, bar.get_y() + bar.get_height() / 2,
             f'{a:.4f}', va='center', fontsize=9, fontweight='bold')

ax2 = fig.add_subplot(gs[0, 2])
comp = [auc_brut_m3, auc_brut_m2]
bars2 = ax2.bar(['M3\n(s_calcif)', 'M2\n(calcif_pct)'], comp,
                color=['#C0392B', '#1D9E75'], alpha=0.85)
ax2.axhline(0.50, color='gray', ls=':', lw=1.2, label='Hasard')
ax2.set_ylim(0.40, max(comp) + 0.08)
ax2.set_ylabel('AUC-ROC')
ax2.set_title('Pouvoir discriminant\nde la source brute', fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3, axis='y')
for bar, a in zip(bars2, comp):
    ax2.text(bar.get_x() + bar.get_width() / 2, a + 0.008,
             f'{a:.4f}', ha='center', fontsize=11, fontweight='bold')

ax3 = fig.add_subplot(gs[1, 0])
bp = ax3.boxplot([g['calcif_pct'].values for g in groupes.values()],
                 tick_labels=list(groupes.keys()), patch_artist=True)
for patch, c in zip(bp['boxes'], ['#2ECC71', '#E74C3C', '#F39C12']):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax3.axhline(SEUIL_M2, color='#C0392B', ls='--', alpha=0.8,
            label=f'Seuil {SEUIL_M2} %')
ax3.set_ylabel('calcif_pct (% de surface)')
ax3.set_title('Signal M2 par groupe\n(surface segmentée)', fontweight='bold')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3, axis='y')

ax4 = fig.add_subplot(gs[1, 1])
bp = ax4.boxplot([g['s_calcif'].values for g in groupes.values()],
                 tick_labels=list(groupes.keys()), patch_artist=True)
for patch, c in zip(bp['boxes'], ['#2ECC71', '#E74C3C', '#F39C12']):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
ax4.set_ylabel('s_calcif (probabilité M3)')
ax4.set_title('Signal M3 par groupe\n(probabilité de classe)', fontweight='bold')
ax4.grid(alpha=0.3, axis='y')

ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
lignes = [[n.replace('\n', ' '), f"{v['auc']:.4f}", f"{v['auc'] - auc_m3:+.4f}"]
          for n, v in resultats.items()]
tab = ax5.table(cellText=lignes,
                colLabels=['Source calcique', 'AUC', 'Δ vs M3'],
                cellLoc='center', loc='center')
tab.auto_set_font_size(False)
tab.set_fontsize(7.5)
tab.scale(1, 1.7)
ax5.set_title('Récapitulatif', fontweight='bold', pad=18)

plt.suptitle(
    f'ÉTUDE 2 — Sources du signal calcique dans la formule V5\n'
    f'M2 : Dice = {DICE_M2} | M3 : F1 = {F1_M3} | n = {len(df)} images NIH',
    fontsize=13, fontweight='bold')

for ext in ('png', 'svg'):
    plt.savefig(f'etude2_sources_calciques.{ext}', dpi=300,
                bbox_inches='tight', facecolor='white')
plt.show()

# ============================================================
# 8. SYNTHÈSE
# ============================================================
print(f"\n{'='*62}")
print("  SYNTHÈSE — ÉTUDE 2")
print(f"{'='*62}\n")

for rang, (nom, v) in enumerate(
        sorted(resultats.items(), key=lambda x: x[1]['auc'], reverse=True), 1):
    marque = '  ← déployée' if nom == NOM_M3 else ''
    print(f"  {rang}. {nom.replace(chr(10), ' '):<30} {v['auc']:.4f}{marque}")

print(f"\n  Un module de segmentation performant sur sa tâche propre")
print(f"  (M2, Dice = {DICE_M2}) ne fournit pas nécessairement le signal")
print(f"  le plus discriminant en aval : la surface calcifiée mesurée")
print(f"  varie de {etendue_m2:.3f} point de pourcentage seulement entre")
print(f"  les groupes cliniques. Le classifieur M3, moins performant sur")
print(f"  sa propre métrique (F1 = {F1_M3}), produit un score de {etendue_m3:.3f}")
print(f"  d'étendue inter-groupes et reste la source retenue.")

print(f"\n  Fichiers générés : etude2_sources_calciques.png / .svg")
